# Qwen2 Baseline vs Qwen3 Checkpoint-250 Evaluation and Inference

Compares exactly two adapters:

- Baseline: `/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_order_refine_v1/runs/20260714_003635/lgt_order_refine/best_adapter`
- Current: `/content/drive/MyDrive/SNU_AI_Challenge/qwen3vl_4b_4task_structured_v1/runs/20260716_065651/qwen3vl_4b_4task/checkpoint-250`

The notebook evaluates both on the fixed tuning/holdout splits, then writes a test submission for the current checkpoint only.

In [ ]:
# 1) Install dependencies, then restart runtime once.
# After restart, run this cell again and continue.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path('/content/.snu_qwen_compare_deps_installed')

if not MARKER.exists():
    packages = [
        'transformers>=4.57.0',
        'accelerate>=0.34.0',
        'bitsandbytes>=0.46.1',
        'peft',
        'qwen-vl-utils',
        'modelscope',
        'hf_transfer',
        'jedi',
        'pandas==2.2.2',
        'safetensors>=0.4.5',
    ]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', *packages])
    MARKER.write_text('ok')
    print('Dependencies installed. Restarting runtime. Run this cell again after restart.')
    os.kill(os.getpid(), 9)
else:
    print('Dependencies already installed. Continue.')

In [ ]:
# 2) Setup, data, and fixed paths
from google.colab import drive
drive.mount('/content/drive')

import ast
import gc
import glob
import itertools
import json
import math
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
import random
import re
import shutil
import zipfile
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig, set_seed
try:
    from transformers import Qwen3VLForConditionalGeneration
except ImportError:
    Qwen3VLForConditionalGeneration = None
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = None
try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    AutoModelForImageTextToText = None
try:
    from transformers import AutoModelForVision2Seq
except ImportError:
    AutoModelForVision2Seq = None
if Qwen2VLForConditionalGeneration is None and AutoModelForVision2Seq is not None:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import PeftModel
from transformers.utils import logging as transformers_logging

warnings.filterwarnings('ignore', category=FutureWarning, module='bitsandbytes')
warnings.filterwarnings('ignore', message='.*The following generation flags are not valid.*')
transformers_logging.set_verbosity_error()

ZIP_PATH = '/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip'
DATA_DIR = '/content/snuaichallenge_data'
TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
TEST_CSV = os.path.join(DATA_DIR, 'test.csv')
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, 'train')
TEST_IMAGE_DIR = os.path.join(DATA_DIR, 'test')

SPLIT_DIR = '/content/drive/MyDrive/SNU_AI_Challenge/id_splits/qwen2vl_lgt_order_refine_20260714_003635'
BASELINE_ADAPTER_DIR = '/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_order_refine_v1/runs/20260714_003635/lgt_order_refine/best_adapter'
QWEN2_BASELINE_STRUCTURED_CACHE_DIR = '/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_multi_hypothesis_refine_v1/shared_cache/baseline_structured_cache'
CURRENT_ADAPTER_DIR = '/content/drive/MyDrive/SNU_AI_Challenge/qwen3vl_4b_4task_structured_v1/runs/20260716_065651/qwen3vl_4b_4task/checkpoint-250'

BASELINE_FALLBACK_MODEL_ID = 'Qwen/Qwen2-VL-2B-Instruct'
CURRENT_FALLBACK_MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
USE_MODELSCOPE_BASE_MODEL = True
DRIVE_MODEL_CACHE_ROOT = '/content/drive/MyDrive/SNU_AI_Challenge/model_cache'

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = '/content/drive/MyDrive/SNU_AI_Challenge/qwen2_vs_qwen3_checkpoint250_compare'
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, RUN_ID)
EVAL_DIR = os.path.join(OUTPUT_DIR, 'eval')
SUBMISSION_DIR = os.path.join(OUTPUT_DIR, 'submissions')
SHARED_PROB_CACHE_DIR = os.path.join(OUTPUT_ROOT, 'shared_probability_cache')
for path in [OUTPUT_ROOT, OUTPUT_DIR, EVAL_DIR, SUBMISSION_DIR, SHARED_PROB_CACHE_DIR]:
    os.makedirs(path, exist_ok=True)

SEED = 42
QUICK_EVAL_ROWS = 50
TUNING_EVAL_ROWS = 150
HOLDOUT_EVAL_ROWS = 150
MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
PAIR_BATCH_SIZE = 2
PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))
ALPHAS = [1.0]
BETAS = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]
GAMMAS = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]

if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall('/content/')

for required in [TRAIN_CSV, TEST_CSV, TRAIN_IMAGE_DIR, TEST_IMAGE_DIR, SPLIT_DIR, BASELINE_ADAPTER_DIR, QWEN2_BASELINE_STRUCTURED_CACHE_DIR, CURRENT_ADAPTER_DIR]:
    assert os.path.exists(required), required

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('output:', OUTPUT_DIR)
print('baseline adapter:', BASELINE_ADAPTER_DIR)
print('current adapter:', CURRENT_ADAPTER_DIR)

In [ ]:
# 3) Data helpers and fixed split loading
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def read_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def save_json(data, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f'Invalid Answer: {answer}')
    return result


def order_to_sequence(answer):
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]


def sequence_to_answer(order):
    answer = [0] * 4
    for rank, image_number in enumerate(order, start=1):
        answer[int(image_number) - 1] = rank
    return answer


def parse_order_prediction(text):
    strict = re.fullmatch(r'\s*\[\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*\]\s*', str(text))
    if strict:
        values = [int(value) for value in strict.groups()]
        return values if sorted(values) == [1, 2, 3, 4] else None
    loose = re.search(r'\[\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*\]', str(text))
    if not loose:
        return None
    values = [int(value) for value in loose.groups()]
    return values if sorted(values) == [1, 2, 3, 4] else None


def row_image_paths(row, image_root):
    sample_id = str(row['Id'])
    return [os.path.join(image_root, sample_id, str(row[f'Input_{i}'])) for i in range(1, 5)]


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert('RGB').copy()


def rows_by_ids(dataframe, ids):
    ids = [str(value) for value in ids]
    subset = dataframe[dataframe['Id'].isin(ids)].copy()
    order = {sample_id: index for index, sample_id in enumerate(ids)}
    subset['_split_order'] = subset['Id'].map(order)
    return subset.sort_values('_split_order').drop(columns=['_split_order']).reset_index(drop=True)


train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df['Id'] = train_df['Id'].astype(str)
test_df['Id'] = test_df['Id'].astype(str)
train_df['Answer_list'] = train_df['Answer'].apply(parse_answer)

quick50_df = rows_by_ids(train_df, load_json(os.path.join(SPLIT_DIR, 'quick50_ids.json')))
tuning150_df = rows_by_ids(train_df, load_json(os.path.join(SPLIT_DIR, 'tuning150_ids.json')))
holdout150_df = rows_by_ids(train_df, load_json(os.path.join(SPLIT_DIR, 'holdout150_ids.json')))

print('quick/tuning/holdout/test:', len(quick50_df), len(tuning150_df), len(holdout150_df), len(test_df))

In [ ]:
# 4) Prompt and model helpers
def task_instruction(example):
    sentence = example['sentence']
    task_type = example['task_type']
    if task_type == 'pairwise':
        return (
            f'Caption:\n{sentence}\n\n'
            'Question: Which image occurs first?\n'
            'If the first image occurs earlier, answer 1.\n'
            'If the second image occurs earlier, answer 2.\n'
            'Answer only 1 or 2.'
        )
    if task_type == 'first':
        return (
            f'Caption:\n{sentence}\n\n'
            'Question: Which image represents the beginning of the story?\n'
            'Answer only the image number from 1 to 4.'
        )
    if task_type == 'last':
        return (
            f'Caption:\n{sentence}\n\n'
            'Question: Which image represents the end of the story?\n'
            'Answer only the image number from 1 to 4.'
        )
    if task_type == 'order':
        return (
            f'Caption:\n{sentence}\n\n'
            'Question: Compare the temporal relation between scenes and identify the likely first and last scenes.\n'
            'Using these cues, determine the complete chronological order.\n'
            'Output only the final ordered list, such as [1, 2, 3, 4].'
        )
    raise ValueError(task_type)


def make_messages(example, include_answer=False):
    content = []
    for idx, _ in enumerate(example['image_paths'], start=1):
        content.append({'type': 'text', 'text': f'\nImage {idx}:'})
        content.append({'type': 'image'})
    content.append({'type': 'text', 'text': '\n\n' + task_instruction(example)})
    messages = [{'role': 'user', 'content': content}]
    if include_answer:
        messages.append({'role': 'assistant', 'content': str(example['target'])})
    return messages


def adapter_model_id(adapter_dir, fallback, override=None):
    if override is not None:
        return override
    config_path = os.path.join(adapter_dir, 'adapter_config.json')
    with open(config_path, 'r', encoding='utf-8') as f:
        config = json.load(f)
    return config.get('base_model_name_or_path') or fallback


def model_cache_is_complete(model_dir):
    if not os.path.exists(os.path.join(model_dir, 'config.json')):
        return False
    has_weight = any(os.path.exists(os.path.join(model_dir, name)) for name in ['model.safetensors.index.json', 'pytorch_model.bin', 'pytorch_model.bin.index.json']) or bool(glob.glob(os.path.join(model_dir, '*.safetensors')))
    has_processor = any(os.path.exists(os.path.join(model_dir, name)) for name in ['preprocessor_config.json', 'processor_config.json', 'tokenizer.json', 'tokenizer_config.json'])
    return bool(has_weight and has_processor)


def ensure_model_path(model_id):
    local_name = model_id.rstrip('/').split('/')[-1]
    drive_dir = os.path.join(DRIVE_MODEL_CACHE_ROOT, local_name)
    if model_cache_is_complete(drive_dir):
        print('Using cached base model:', drive_dir)
        return drive_dir
    if os.path.isdir(model_id) or not USE_MODELSCOPE_BASE_MODEL:
        print('Using model id/path:', model_id)
        return model_id
    print('Base model cache not found. Downloading via ModelScope:', model_id)
    from modelscope import snapshot_download as modelscope_snapshot_download
    downloaded = modelscope_snapshot_download(model_id, cache_dir='/content/modelscope_cache')
    os.makedirs(os.path.dirname(drive_dir), exist_ok=True)
    if not model_cache_is_complete(drive_dir):
        tmp = drive_dir + '.tmp'
        if os.path.exists(tmp):
            shutil.rmtree(tmp)
        shutil.copytree(downloaded, tmp, dirs_exist_ok=True)
        if os.path.exists(drive_dir):
            shutil.rmtree(drive_dir)
        os.replace(tmp, drive_dir)
    return drive_dir


def load_model_class(model_id):
    lower = str(model_id).lower()
    if 'qwen3' in lower and Qwen3VLForConditionalGeneration is not None:
        return Qwen3VLForConditionalGeneration
    if 'qwen2' in lower and Qwen2VLForConditionalGeneration is not None:
        return Qwen2VLForConditionalGeneration
    if AutoModelForImageTextToText is not None:
        return AutoModelForImageTextToText
    if AutoModelForVision2Seq is not None:
        return AutoModelForVision2Seq
    raise ImportError('No compatible vision-language model class found.')


def sanitize_generation_config(active_model):
    generation_config = active_model.generation_config
    generation_config.do_sample = False
    generation_config.temperature = None
    generation_config.top_p = None
    generation_config.top_k = None
    generation_config.num_beams = 1
    return active_model


def make_bnb_config(spec):
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=spec['torch_dtype'],
    )

MODEL_SPECS = [
    {'name': 'qwen2_baseline_best_adapter', 'adapter_dir': BASELINE_ADAPTER_DIR, 'fallback_model_id': BASELINE_FALLBACK_MODEL_ID, 'model_id_override': BASELINE_FALLBACK_MODEL_ID, 'torch_dtype': torch.float16, 'pairwise_bidirectional': False, 'eval_direct': False, 'cache_only': True},
    {'name': 'qwen3_run20260716_checkpoint250', 'adapter_dir': CURRENT_ADAPTER_DIR, 'fallback_model_id': CURRENT_FALLBACK_MODEL_ID, 'model_id_override': None, 'torch_dtype': torch.bfloat16, 'pairwise_bidirectional': True, 'eval_direct': False},
]
for spec in MODEL_SPECS:
    spec['model_id'] = adapter_model_id(spec['adapter_dir'], spec['fallback_model_id'], override=spec.get('model_id_override'))
    if spec.get('cache_only', False):
        spec['model_path'] = None
        spec['local_files_only'] = True
    else:
        spec['model_path'] = ensure_model_path(spec['model_id'])
        spec['local_files_only'] = os.path.isdir(spec['model_path'])
MODEL_SPECS

In [ ]:
# 5) Evaluation and decoding functions
def make_eval_example(row, task_type, pair=None):
    answer = [int(value) for value in row.get('Answer_list', [1, 2, 3, 4])]
    image_paths = row_image_paths(row, TRAIN_IMAGE_DIR)
    example = {
        'sample_id': str(row['Id']),
        'sentence': '' if pd.isna(row['Sentence']) else str(row['Sentence']),
        'answer': answer,
        'order': order_to_sequence(answer),
        'image_paths': image_paths,
        'task_type': task_type,
        'target': '1',
    }
    if task_type == 'pairwise':
        a, b = pair
        example['image_paths'] = [image_paths[a - 1], image_paths[b - 1]]
    return example


def make_test_example(row, task_type, pair=None):
    image_paths = row_image_paths(row, TEST_IMAGE_DIR)
    example = {
        'sample_id': str(row['Id']),
        'sentence': '' if pd.isna(row['Sentence']) else str(row['Sentence']),
        'answer': [1, 2, 3, 4],
        'order': [1, 2, 3, 4],
        'image_paths': image_paths,
        'task_type': task_type,
        'target': '1',
    }
    if task_type == 'pairwise':
        a, b = pair
        example['image_paths'] = [image_paths[a - 1], image_paths[b - 1]]
    return example


def model_device(active_model):
    return next(active_model.parameters()).device


def digit_token_ids(processor):
    ids = {}
    for digit in [1, 2, 3, 4]:
        token_ids = processor.tokenizer.encode(str(digit), add_special_tokens=False)
        if len(token_ids) != 1:
            raise ValueError(f'Digit {digit} tokenized to {token_ids}')
        ids[digit] = token_ids[0]
    return ids


def load_eval_bundle(spec):
    processor = AutoProcessor.from_pretrained(
        spec['model_path'],
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
        local_files_only=spec['local_files_only'],
        trust_remote_code=True,
    )
    processor.tokenizer.padding_side = 'right'
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    model_cls = load_model_class(spec['model_id'])
    base = model_cls.from_pretrained(
        spec['model_path'],
        quantization_config=make_bnb_config(spec),
        torch_dtype=spec['torch_dtype'],
        device_map='auto',
        local_files_only=spec['local_files_only'],
        trust_remote_code=True,
    )
    loaded = PeftModel.from_pretrained(base, spec['adapter_dir'], is_trainable=False)
    sanitize_generation_config(loaded)
    loaded.eval()
    return {'model': loaded, 'processor': processor, 'digit_token_ids': digit_token_ids(processor)}


EVAL_BUNDLES = {}


def get_eval_bundle(spec):
    name = spec['name']
    if name not in EVAL_BUNDLES:
        EVAL_BUNDLES[name] = load_eval_bundle(spec)
    return EVAL_BUNDLES[name]


@torch.no_grad()
def score_digit_candidates(bundle, example, candidates):
    active_model = bundle['model']
    processor = bundle['processor']
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = 'right'
    text = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example['image_paths']]
    inputs = processor(text=[text], images=[images], return_tensors='pt')
    inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
    outputs = active_model(**inputs)
    last_pos = int(inputs['attention_mask'][0].sum().item()) - 1
    logits = outputs.logits[0, last_pos]
    token_ids = [bundle['digit_token_ids'][int(candidate)] for candidate in candidates]
    probs = torch.softmax(logits[token_ids].float(), dim=-1).detach().cpu().numpy()
    processor.tokenizer.padding_side = old_padding_side
    return {int(candidate): float(prob) for candidate, prob in zip(candidates, probs)}


@torch.no_grad()
def score_digit_candidate_batch(bundle, examples, candidates_per_example):
    active_model = bundle['model']
    processor = bundle['processor']
    old_padding_side = processor.tokenizer.padding_side
    try:
        processor.tokenizer.padding_side = 'right'
        texts = [processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True) for example in examples]
        image_cache = {}

        def cached_load(path):
            if path not in image_cache:
                image_cache[path] = load_rgb(path)
            return image_cache[path]

        images = [[cached_load(path) for path in example['image_paths']] for example in examples]
        inputs = processor(text=texts, images=images, padding=True, return_tensors='pt')
        inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
        outputs = active_model(**inputs)
        results = []
        for row_index, candidates in enumerate(candidates_per_example):
            last_pos = int(inputs['attention_mask'][row_index].sum().item()) - 1
            logits = outputs.logits[row_index, last_pos]
            token_ids = [bundle['digit_token_ids'][int(candidate)] for candidate in candidates]
            probs = torch.softmax(logits[token_ids].float(), dim=-1).detach().cpu().numpy()
            results.append({int(candidate): float(prob) for candidate, prob in zip(candidates, probs)})
        return results
    finally:
        processor.tokenizer.padding_side = old_padding_side


def score_tasks_batched(bundle, row, maker, bidirectional=True):
    endpoint_probs = score_digit_candidate_batch(
        bundle,
        [maker(row, 'first'), maker(row, 'last')],
        [[1, 2, 3, 4], [1, 2, 3, 4]],
    )
    first_probs = {str(k): v for k, v in endpoint_probs[0].items()}
    last_probs = {str(k): v for k, v in endpoint_probs[1].items()}

    forward_examples = []
    forward_candidates = []
    for first_index, second_index in PAIR_INDICES:
        a, b = first_index + 1, second_index + 1
        forward_examples.append(maker(row, 'pairwise', pair=(a, b)))
        forward_candidates.append([1, 2])
    forward_probs = []
    for start in range(0, len(forward_examples), PAIR_BATCH_SIZE):
        forward_probs.extend(score_digit_candidate_batch(
            bundle,
            forward_examples[start:start + PAIR_BATCH_SIZE],
            forward_candidates[start:start + PAIR_BATCH_SIZE],
        ))

    reverse_probs = None
    if bidirectional:
        reverse_examples = []
        reverse_candidates = []
        for first_index, second_index in PAIR_INDICES:
            a, b = first_index + 1, second_index + 1
            reverse_examples.append(maker(row, 'pairwise', pair=(b, a)))
            reverse_candidates.append([1, 2])
        reverse_probs = []
        for start in range(0, len(reverse_examples), PAIR_BATCH_SIZE):
            reverse_probs.extend(score_digit_candidate_batch(
                bundle,
                reverse_examples[start:start + PAIR_BATCH_SIZE],
                reverse_candidates[start:start + PAIR_BATCH_SIZE],
            ))

    pair_probs = {}
    for idx, (first_index, second_index) in enumerate(PAIR_INDICES):
        a, b = first_index + 1, second_index + 1
        p_a_before_b = float(forward_probs[idx][1])
        if bidirectional:
            p_a_before_b = 0.5 * (p_a_before_b + float(reverse_probs[idx][2]))
        pair_probs[f'{a}>{b}'] = float(p_a_before_b)
        pair_probs[f'{b}>{a}'] = float(1.0 - p_a_before_b)
    return first_probs, last_probs, pair_probs


def score_test_tasks_batched(bundle, row, bidirectional=True):
    return score_tasks_batched(bundle, row, make_test_example, bidirectional=bidirectional)


def score_eval_tasks_batched(bundle, row, bidirectional=True):
    return score_tasks_batched(bundle, row, make_eval_example, bidirectional=bidirectional)


def score_pair_probability(bundle, row, a, b, bidirectional=True, is_test=False):
    maker = make_test_example if is_test else make_eval_example
    forward = score_digit_candidates(bundle, maker(row, 'pairwise', pair=(a, b)), [1, 2])
    p_a_before_b = float(forward[1])
    if bidirectional:
        reverse = score_digit_candidates(bundle, maker(row, 'pairwise', pair=(b, a)), [1, 2])
        p_a_before_b = 0.5 * (p_a_before_b + float(reverse[2]))
    return p_a_before_b


@torch.no_grad()
def generate_order(bundle, row, is_test=False, max_new_tokens=16):
    active_model = bundle['model']
    processor = bundle['processor']
    maker = make_test_example if is_test else make_eval_example
    example = maker(row, 'order')
    text = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example['image_paths']]
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = 'left'
    inputs = processor(text=[text], images=[images], return_tensors='pt')
    inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
    generated = active_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None)
    new_tokens = generated[:, inputs['input_ids'].shape[1]:]
    output = processor.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]
    processor.tokenizer.padding_side = old_padding_side
    return parse_order_prediction(output), output


def order_ranks(order):
    return {int(image_number): position for position, image_number in enumerate(order)}


def pair_accuracy_from_orders(pred_order, gold_order):
    pred_ranks = order_ranks(pred_order)
    gold_ranks = order_ranks(gold_order)
    return np.mean([
        (pred_ranks[a] < pred_ranks[b]) == (gold_ranks[a] < gold_ranks[b])
        for a, b in itertools.combinations([1, 2, 3, 4], 2)
    ])


def order_metric_row(pred_order, gold_order):
    if pred_order is None:
        return {'exact_match': 0.0, 'pair_accuracy': 0.0, 'position_accuracy': 0.0, 'valid_output': 0.0}
    return {
        'exact_match': float(pred_order == gold_order),
        'pair_accuracy': float(pair_accuracy_from_orders(pred_order, gold_order)),
        'position_accuracy': float(np.mean([p == g for p, g in zip(pred_order, gold_order)])),
        'valid_output': 1.0,
    }


def structured_score(sample, order, alpha, beta, gamma):
    eps = 1e-12
    pair_score = np.mean([
        math.log(float(sample['pair_probs'][f'{order[i]}>{order[j]}']) + eps)
        for i in range(4)
        for j in range(i + 1, 4)
    ])
    first_score = math.log(float(sample['first_probs'][str(order[0])]) + eps)
    last_score = math.log(float(sample['last_probs'][str(order[-1])]) + eps)
    return alpha * pair_score + beta * first_score + gamma * last_score


def decode_structured(sample, alpha, beta, gamma):
    return list(max(PERMUTATIONS, key=lambda order: structured_score(sample, order, alpha, beta, gamma)))

In [ ]:
# 6) Probability cache, metrics, and weight search
def find_external_qwen2_baseline_cache(rows, tag):
    expected_ids = rows['Id'].astype(str).tolist()
    candidates = sorted(glob.glob(os.path.join(QWEN2_BASELINE_STRUCTURED_CACHE_DIR, '*', f'{tag}_baseline.jsonl')))
    for path in candidates:
        records = read_jsonl(path)
        if [str(record.get('sample_id')) for record in records] == expected_ids:
            return path, records
    raise FileNotFoundError(
        f'No matching Qwen2 baseline cache for {tag}. Expected {len(expected_ids)} ids under {QWEN2_BASELINE_STRUCTURED_CACHE_DIR}. '
        'Run the Qwen2-only 8/10/14 eval notebook once, or copy its baseline_structured_cache directory.'
    )


def convert_external_qwen2_baseline_records(records):
    converted = []
    for record in records:
        gold = [int(x) for x in record['gold_order']]
        pred = [int(x) for x in record.get('baseline_order', record.get('candidate_orders', [{}])[0].get('order'))]
        pair_probs = {str(k): float(v) for k, v in record['pair_probs'].items()}
        first_probs = {str(k): float(v) for k, v in record['first_probs'].items()}
        last_probs = {str(k): float(v) for k, v in record['last_probs'].items()}
        pair_correct = []
        answer_rank = {image: idx for idx, image in enumerate(gold)}
        for first_index, second_index in PAIR_INDICES:
            a, b = first_index + 1, second_index + 1
            p_a_before_b = float(pair_probs[f'{a}>{b}'])
            pred_first = a if p_a_before_b >= 0.5 else b
            gold_first = a if answer_rank[a] < answer_rank[b] else b
            pair_correct.append(int(pred_first == gold_first))
        converted.append({
            'sample_id': str(record['sample_id']),
            'gold_order': gold,
            'first_probs': first_probs,
            'last_probs': last_probs,
            'pair_probs': pair_probs,
            'direct_order': None,
            'direct_text': '',
            'pairwise_accuracy': float(np.mean(pair_correct)),
            'first_accuracy': float(max(first_probs, key=first_probs.get) == gold[0]),
            'last_accuracy': float(max(last_probs, key=last_probs.get) == gold[-1]),
            'external_baseline_order': pred,
        })
    return converted


def validate_qwen2_cache_against_rows(records, rows, tag):
    expected_ids = rows['Id'].astype(str).tolist()
    actual_ids = [str(record.get('sample_id')) for record in records]
    assert actual_ids == expected_ids, {
        'tag': tag,
        'expected_count': len(expected_ids),
        'actual_count': len(actual_ids),
        'first_expected': expected_ids[:5],
        'first_actual': actual_ids[:5],
    }
    row_by_id = {str(row['Id']): row for _, row in rows.iterrows()}
    required = {'sample_id', 'gold_order', 'first_probs', 'last_probs', 'pair_probs'}
    for idx, record in enumerate(records):
        missing = sorted(required - set(record.keys()))
        assert not missing, {'tag': tag, 'idx': idx, 'sample_id': record.get('sample_id'), 'missing': missing}
        sample_id = str(record['sample_id'])
        gold_from_csv = order_to_sequence([int(value) for value in row_by_id[sample_id]['Answer_list']])
        gold_from_cache = [int(value) for value in record['gold_order']]
        assert gold_from_cache == gold_from_csv, {'tag': tag, 'sample_id': sample_id, 'cache_gold': gold_from_cache, 'csv_gold': gold_from_csv}
        for key in ['first_probs', 'last_probs']:
            assert sorted(str(k) for k in record[key].keys()) == ['1', '2', '3', '4'], {'tag': tag, 'sample_id': sample_id, 'bad_key': key, 'keys': record[key].keys()}
        pair_keys = set(record['pair_probs'].keys())
        expected_pair_keys = {f'{a}>{b}' for a in [1, 2, 3, 4] for b in [1, 2, 3, 4] if a != b}
        assert expected_pair_keys <= pair_keys, {'tag': tag, 'sample_id': sample_id, 'missing_pair_keys': sorted(expected_pair_keys - pair_keys)[:5]}
    print(f'[CACHE OK] {tag}: {len(records)} rows match split ids, gold_order, and probability keys')


def verify_available_qwen2_baseline_caches(strict=False):
    checks = [('quick50', quick50_df), (f'tuning{len(tuning150_df)}', tuning150_df), (f'holdout{len(holdout150_df)}', holdout150_df)]
    found = {}
    for tag, rows in checks:
        try:
            path, records = find_external_qwen2_baseline_cache(rows, tag)
            validate_qwen2_cache_against_rows(records, rows, tag)
            found[tag] = path
        except FileNotFoundError as exc:
            found[tag] = None
            print(f'[CACHE MISSING] {tag}: {exc}')
            if strict:
                raise
    return found


def extract_probability_cache(spec, rows, tag):
    name = spec['name']
    cache_path = os.path.join(SHARED_PROB_CACHE_DIR, f'{name}_{tag}_task_probability_cache.json')
    if os.path.exists(cache_path):
        print('[SKIP]', cache_path)
        return load_json(cache_path)

    if spec.get('cache_only', False):
        external_path, external_records = find_external_qwen2_baseline_cache(rows, tag)
        print('[QWEN2 CACHE]', external_path)
        validate_qwen2_cache_against_rows(external_records, rows, tag)
        records = convert_external_qwen2_baseline_records(external_records)
        save_json(records, cache_path)
        shutil.copy2(cache_path, os.path.join(EVAL_DIR, os.path.basename(cache_path)))
        return records

    bundle = get_eval_bundle(spec)
    records = []
    for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f'{name} {tag} probs'):
        answer = [int(value) for value in row['Answer_list']]
        gold_order = order_to_sequence(answer)
        first_probs, last_probs, pair_probs = score_eval_tasks_batched(
            bundle,
            row,
            bidirectional=bool(spec.get('pairwise_bidirectional', True)),
        )
        pair_correct = []
        for first_index, second_index in PAIR_INDICES:
            a, b = first_index + 1, second_index + 1
            p_a_before_b = float(pair_probs[f'{a}>{b}'])
            pred_first = a if p_a_before_b >= 0.5 else b
            gold_first = a if answer[first_index] < answer[second_index] else b
            pair_correct.append(int(pred_first == gold_first))

        if spec.get('eval_direct', False):
            direct_order, direct_text = generate_order(bundle, row)
        else:
            direct_order, direct_text = None, ''
        records.append({
            'sample_id': str(row['Id']),
            'gold_order': gold_order,
            'first_probs': {str(k): v for k, v in first_probs.items()},
            'last_probs': {str(k): v for k, v in last_probs.items()},
            'pair_probs': pair_probs,
            'direct_order': direct_order,
            'direct_text': direct_text,
            'pairwise_accuracy': float(np.mean(pair_correct)),
            'first_accuracy': float(max(first_probs, key=first_probs.get) == gold_order[0]),
            'last_accuracy': float(max(last_probs, key=last_probs.get) == gold_order[-1]),
        })

    save_json(records, cache_path)
    shutil.copy2(cache_path, os.path.join(EVAL_DIR, os.path.basename(cache_path)))
    return records


def evaluate_decoding(samples, alpha=1.0, beta=1.0, gamma=1.0, mode='structured'):
    rows = []
    for sample in samples:
        gold = [int(value) for value in sample['gold_order']]
        pred = sample['direct_order'] if mode == 'direct' else decode_structured(sample, alpha, beta, gamma)
        metric = order_metric_row(pred, gold)
        metric.update({
            'sample_id': sample['sample_id'],
            'pred_order': pred,
            'gold_order': gold,
            'pairwise_accuracy': sample['pairwise_accuracy'],
            'first_accuracy': sample['first_accuracy'],
            'last_accuracy': sample['last_accuracy'],
            'first_last_both_correct': float(sample['first_accuracy'] == 1.0 and sample['last_accuracy'] == 1.0),
        })
        rows.append(metric)
    df = pd.DataFrame(rows)
    summary = {
        'exact_match': df['exact_match'].mean(),
        'pair_accuracy': df['pair_accuracy'].mean(),
        'position_accuracy': df['position_accuracy'].mean(),
        'valid_output_rate': df['valid_output'].mean(),
        'mean_pairwise_accuracy': df['pairwise_accuracy'].mean(),
        'mean_first_accuracy': df['first_accuracy'].mean(),
        'mean_last_accuracy': df['last_accuracy'].mean(),
        'first_last_both_correct_rate': df['first_last_both_correct'].mean(),
        'exact_given_first_last_correct': df.loc[df['first_last_both_correct'] == 1.0, 'exact_match'].mean() if (df['first_last_both_correct'] == 1.0).any() else np.nan,
    }
    return summary, df


def grid_search(samples, model_name, tag):
    path = os.path.join(SHARED_PROB_CACHE_DIR, f'{model_name}_{tag}_decoding_weight_search.csv')
    if os.path.exists(path):
        return pd.read_csv(path)
    rows = []
    for alpha, beta, gamma in itertools.product(ALPHAS, BETAS, GAMMAS):
        summary, _ = evaluate_decoding(samples, alpha=alpha, beta=beta, gamma=gamma, mode='structured')
        summary.update({'model': model_name, 'tag': tag, 'alpha': alpha, 'beta': beta, 'gamma': gamma, 'decoding': 'pair_first_last'})
        rows.append(summary)
    df = pd.DataFrame(rows).sort_values(['exact_match', 'pair_accuracy', 'position_accuracy'], ascending=False).reset_index(drop=True)
    df.to_csv(path, index=False)
    shutil.copy2(path, os.path.join(EVAL_DIR, os.path.basename(path)))
    return df

In [ ]:
# 6.5) Verify external Qwen2 baseline caches before running comparisons
qwen2_cache_paths = verify_available_qwen2_baseline_caches()
qwen2_cache_paths


In [ ]:
# 7) Evaluate both models on tuning, select best decoding per model, then report holdout
selection_rows = []
holdout_rows = []
best_by_model = {}

for spec in MODEL_SPECS:
    name = spec['name']
    tuning_samples = extract_probability_cache(spec, tuning150_df, tag=f'tuning{len(tuning150_df)}')

    if spec.get('eval_direct', False):
        direct_summary, direct_predictions = evaluate_decoding(tuning_samples, mode='direct')
        direct_predictions.to_csv(os.path.join(EVAL_DIR, f'{name}_tuning_direct_predictions.csv'), index=False)
        direct_summary.update({'model': name, 'split': 'tuning150', 'decoding': 'direct', 'alpha': np.nan, 'beta': np.nan, 'gamma': np.nan})
        selection_rows.append(direct_summary)

    search = grid_search(tuning_samples, name, tag=f'tuning{len(tuning150_df)}')
    best_structured = search.iloc[0].to_dict()
    best_structured.update({'split': 'tuning150'})
    selection_rows.append(best_structured)

    selection_df_partial = pd.DataFrame(selection_rows)
    selection_df_partial.to_csv(os.path.join(EVAL_DIR, 'tuning_selection_metrics.csv'), index=False)

selection_df = pd.DataFrame(selection_rows).sort_values(['model', 'exact_match', 'pair_accuracy', 'position_accuracy'], ascending=[True, False, False, False]).reset_index(drop=True)
display(selection_df)

for spec in MODEL_SPECS:
    name = spec['name']
    model_selection = selection_df[selection_df['model'].eq(name)].sort_values(['exact_match', 'pair_accuracy', 'position_accuracy'], ascending=False).reset_index(drop=True)
    best = model_selection.iloc[0].to_dict()
    best_by_model[name] = best
    print('BEST TUNING', name, best)

    try:
        holdout_samples = extract_probability_cache(spec, holdout150_df, tag=f'holdout{len(holdout150_df)}')
    except FileNotFoundError as exc:
        if spec.get('cache_only', False):
            print(f'[SKIP HOLDOUT] {name}: {exc}')
            holdout_rows.append({
                'model': name,
                'split': 'holdout150',
                'status': 'skipped_missing_qwen2_cache',
                'decoding': best['decoding'],
                'alpha': None if pd.isna(best.get('alpha', np.nan)) else float(best['alpha']),
                'beta': None if pd.isna(best.get('beta', np.nan)) else float(best['beta']),
                'gamma': None if pd.isna(best.get('gamma', np.nan)) else float(best['gamma']),
                'tuning_exact_match': float(best['exact_match']),
                'tuning_pair_accuracy': float(best['pair_accuracy']),
                'tuning_position_accuracy': float(best['position_accuracy']),
                'adapter_dir': spec['adapter_dir'],
                'model_id': spec['model_id'],
                'exact_match': np.nan,
                'pair_accuracy': np.nan,
                'position_accuracy': np.nan,
            })
            continue
        raise
    if best['decoding'] == 'direct':
        holdout_summary, holdout_predictions = evaluate_decoding(holdout_samples, mode='direct')
    else:
        holdout_summary, holdout_predictions = evaluate_decoding(
            holdout_samples,
            alpha=float(best['alpha']),
            beta=float(best['beta']),
            gamma=float(best['gamma']),
            mode='structured',
        )
    holdout_summary.update({
        'model': name,
        'split': 'holdout150',
        'decoding': best['decoding'],
        'alpha': None if pd.isna(best.get('alpha', np.nan)) else float(best['alpha']),
        'beta': None if pd.isna(best.get('beta', np.nan)) else float(best['beta']),
        'gamma': None if pd.isna(best.get('gamma', np.nan)) else float(best['gamma']),
        'tuning_exact_match': float(best['exact_match']),
        'tuning_pair_accuracy': float(best['pair_accuracy']),
        'tuning_position_accuracy': float(best['position_accuracy']),
        'adapter_dir': spec['adapter_dir'],
        'model_id': spec['model_id'],
    })
    holdout_predictions.to_csv(os.path.join(EVAL_DIR, f'{name}_holdout_predictions_selected.csv'), index=False)
    holdout_rows.append(holdout_summary)

holdout_df = pd.DataFrame(holdout_rows).sort_values(['exact_match', 'pair_accuracy', 'position_accuracy'], ascending=False).reset_index(drop=True)
holdout_df.to_csv(os.path.join(EVAL_DIR, 'holdout_metrics_comparison.csv'), index=False)
save_json(best_by_model, os.path.join(EVAL_DIR, 'best_decoding_by_model.json'))
display(holdout_df)

In [ ]:
# 8) Optional quick split sanity check for both models
quick_rows = []
for spec in MODEL_SPECS:
    name = spec['name']
    samples = extract_probability_cache(spec, quick50_df, tag=f'quick{len(quick50_df)}')
    if spec.get('eval_direct', False):
        direct_summary, _ = evaluate_decoding(samples, mode='direct')
        direct_summary.update({'model': name, 'split': 'quick50', 'decoding': 'direct', 'alpha': np.nan, 'beta': np.nan, 'gamma': np.nan})
        quick_rows.append(direct_summary)
    search = grid_search(samples, name, tag=f'quick{len(quick50_df)}')
    best_structured = search.iloc[0].to_dict()
    best_structured.update({'split': 'quick50'})
    quick_rows.append(best_structured)

quick_df = pd.DataFrame(quick_rows).sort_values(['exact_match', 'pair_accuracy', 'position_accuracy'], ascending=False).reset_index(drop=True)
quick_df.to_csv(os.path.join(EVAL_DIR, 'quick_metrics_comparison.csv'), index=False)
display(quick_df)

In [ ]:
# 9) Test inference for current checkpoint only
def infer_one_submission(spec, best_config):
    name = spec['name']
    bundle = get_eval_bundle(spec)
    submission_rows = []
    test_cache = []
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f'{name} test inference'):
        if best_config['decoding'] == 'direct':
            pred_order, text = generate_order(bundle, row, is_test=True)
            if pred_order is None:
                first_probs, last_probs, pair_probs = score_test_tasks_batched(
                    bundle,
                    row,
                    bidirectional=bool(spec.get('pairwise_bidirectional', True)),
                )
                fallback_sample = {
                    'sample_id': str(row['Id']),
                    'first_probs': {str(k): v for k, v in first_probs.items()},
                    'last_probs': {str(k): v for k, v in last_probs.items()},
                    'pair_probs': pair_probs,
                }
                pred_order = decode_structured(fallback_sample, 1.0, 1.0, 1.0)
                test_cache.append(fallback_sample | {'pred_order': pred_order, 'direct_text': text, 'used_fallback': True})
            else:
                test_cache.append({'sample_id': str(row['Id']), 'pred_order': pred_order, 'direct_text': text, 'used_fallback': False})
        else:
            first_probs, last_probs, pair_probs = score_test_tasks_batched(
                bundle,
                row,
                bidirectional=bool(spec.get('pairwise_bidirectional', True)),
            )
            sample = {
                'sample_id': str(row['Id']),
                'first_probs': {str(k): v for k, v in first_probs.items()},
                'last_probs': {str(k): v for k, v in last_probs.items()},
                'pair_probs': pair_probs,
            }
            pred_order = decode_structured(sample, float(best_config['alpha']), float(best_config['beta']), float(best_config['gamma']))
            test_cache.append(sample | {'pred_order': pred_order})

        submission_rows.append({'Id': str(row['Id']), 'Answer': str(sequence_to_answer(pred_order))})

    submission = pd.DataFrame(submission_rows)
    submit_path = os.path.join(SUBMISSION_DIR, f'submission_{name}.csv')
    cache_path = os.path.join(EVAL_DIR, f'{name}_test_probability_cache.json')
    submission.to_csv(submit_path, index=False)
    save_json(test_cache, cache_path)
    print('saved:', submit_path)
    return submission


CURRENT_SPEC = next(spec for spec in MODEL_SPECS if spec['name'] == 'qwen3_run20260716_checkpoint250')
current_submission = infer_one_submission(CURRENT_SPEC, best_by_model[CURRENT_SPEC['name']])
display(current_submission.head())